# Exploratory Data Analysis with Pandas and Matplotlib

This notebook presents a complete exploratory data analysis workflow using a realistic synthetic retail-sales dataset.

## Learning objectives

After completing this notebook, students should be able to:

- define exploratory data analysis and explain its role;
- inspect the structure and quality of a dataset;
- summarize numerical and categorical variables;
- analyze distributions using descriptive statistics and plots;
- compare numerical and categorical features;
- use grouping, aggregation, and pivot tables;
- identify potential outliers;
- interpret correlations appropriately;
- prepare a concise data-quality report;
- communicate evidence-based analytical findings.

The dataset is generated inside the notebook, so no external file is required.

## 1. Import the required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 2. Create a realistic retail-sales dataset

The dataset contains order-level information such as:

- order date;
- sales region;
- product category;
- unit price;
- quantity;
- discount;
- customer satisfaction;
- delivery time;
- returned-order status.

A few missing values, duplicate rows, and potential outliers are inserted intentionally.

In [ ]:
n = 240

regions = np.array(["North", "South", "East", "West"])
categories = np.array(["Electronics", "Office", "Home", "Accessories"])
channels = np.array(["Online", "Store", "Partner"])

df = pd.DataFrame({
    "order_id": np.arange(5001, 5001 + n),
    "order_date": pd.date_range("2025-01-01", periods=n, freq="D"),
    "region": rng.choice(regions, size=n, p=[0.28, 0.24, 0.26, 0.22]),
    "category": rng.choice(categories, size=n, p=[0.30, 0.25, 0.25, 0.20]),
    "channel": rng.choice(channels, size=n, p=[0.50, 0.35, 0.15]),
    "unit_price": np.round(rng.lognormal(mean=4.6, sigma=0.55, size=n), 2),
    "quantity": rng.integers(1, 8, size=n),
    "discount": np.round(rng.choice([0, 0.05, 0.10, 0.15, 0.20], size=n, p=[0.28, 0.22, 0.22, 0.18, 0.10]), 2),
    "customer_rating": np.round(np.clip(rng.normal(4.1, 0.65, size=n), 1, 5), 1),
    "delivery_days": np.clip(np.round(rng.normal(4.5, 1.7, size=n)), 1, 12).astype(int),
    "returned": rng.choice(["No", "Yes"], size=n, p=[0.88, 0.12])
})

category_multiplier = {
    "Electronics": 1.8,
    "Office": 0.8,
    "Home": 1.2,
    "Accessories": 0.5
}

df["unit_price"] = (
    df["unit_price"] *
    df["category"].map(category_multiplier)
).round(2)

df["sales"] = (
    df["unit_price"] *
    df["quantity"] *
    (1 - df["discount"])
).round(2)

df.loc[[12, 57, 105, 171], "customer_rating"] = np.nan
df.loc[[22, 88, 144], "delivery_days"] = np.nan
df.loc[[33, 120], "region"] = np.nan
df.loc[199, "unit_price"] *= 8
df.loc[199, "sales"] = round(
    df.loc[199, "unit_price"] *
    df.loc[199, "quantity"] *
    (1 - df.loc[199, "discount"]),
    2
)

df = pd.concat([df, df.iloc[[15]]], ignore_index=True)

df.head()

## 3. What is exploratory data analysis?

Exploratory data analysis, or EDA, is the process of examining data before formal modeling or hypothesis testing.

EDA helps analysts:

- understand dataset structure;
- detect quality problems;
- identify patterns and relationships;
- discover unusual observations;
- generate questions and hypotheses;
- choose suitable statistical or machine-learning methods.

EDA is investigative. It does not replace formal validation, but it guides later analysis.

## 4. Inspect the dataset structure

In [ ]:
print("First five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())

print("\nRandom sample:")
display(df.sample(5, random_state=RANDOM_SEED))

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
df.info()

### Interpretation

The inspection shows:

- each row represents one order;
- there are numerical, categorical, date, and binary variables;
- some columns contain missing values;
- the dataset contains one intentionally duplicated row;
- `delivery_days` became a floating-point column because it contains missing values.

## 5. Check missing values and duplicates

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

missing_report

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
display(df[df.duplicated(keep=False)])

Missingness is limited in this teaching dataset. During EDA, it is acceptable to preserve missing values temporarily while identifying their pattern and importance.

## 6. Generate descriptive statistics

In [ ]:
df.describe().T

In [ ]:
df.describe(include=["object"]).T

### Interpreting common descriptive statistics

- **Mean:** arithmetic average; sensitive to extreme values.
- **Median:** middle value; more robust to outliers.
- **Standard deviation:** typical spread around the mean.
- **Minimum and maximum:** smallest and largest observed values.
- **Quartiles:** divide the ordered data into four parts.
- **Interquartile range:** difference between the third and first quartiles.

A large difference between the mean and median may indicate skewness or influential outliers.

In [ ]:
numeric_summary = pd.DataFrame({
    "mean": df[["unit_price", "quantity", "discount", "customer_rating", "delivery_days", "sales"]].mean(),
    "median": df[["unit_price", "quantity", "discount", "customer_rating", "delivery_days", "sales"]].median(),
    "std": df[["unit_price", "quantity", "discount", "customer_rating", "delivery_days", "sales"]].std(),
    "min": df[["unit_price", "quantity", "discount", "customer_rating", "delivery_days", "sales"]].min(),
    "max": df[["unit_price", "quantity", "discount", "customer_rating", "delivery_days", "sales"]].max()
})

numeric_summary

## 7. Analyze numerical distributions

### 7.1 Histogram of sales

A histogram shows how frequently observations fall within numerical intervals.

In [ ]:
df["sales"].plot(
    kind="hist",
    bins=25,
    title="Distribution of order sales",
    xlabel="Sales"
)
plt.show()

The sales distribution is right-skewed. Most orders have moderate sales values, while a smaller number of orders are much larger.

### 7.2 Box plot of sales

A box plot summarizes the median, quartiles, spread, and potential outliers.

In [ ]:
df["sales"].plot(
    kind="box",
    title="Box plot of order sales",
    ylabel="Sales"
)
plt.show()

The points beyond the upper whisker are candidates for review. They are not automatically errors because legitimate high-value orders may exist.

### 7.3 Compare mean and median

In [ ]:
sales_mean = df["sales"].mean()
sales_median = df["sales"].median()

print(f"Mean sales: {sales_mean:.2f}")
print(f"Median sales: {sales_median:.2f}")
print(f"Difference: {sales_mean - sales_median:.2f}")

Because the mean is influenced by large sales values, it is expected to be higher than the median in a right-skewed distribution.

## 8. Analyze categorical variables

In [ ]:
region_counts = df["region"].value_counts(dropna=False)
region_percentages = df["region"].value_counts(dropna=False, normalize=True).mul(100).round(2)

pd.DataFrame({
    "count": region_counts,
    "percentage": region_percentages
})

In [ ]:
df["category"].value_counts().plot(
    kind="bar",
    title="Orders by product category",
    xlabel="Category",
    ylabel="Number of orders"
)
plt.xticks(rotation=0)
plt.show()

Bar charts are generally more effective than pie charts for comparing category sizes because lengths are easier to compare than angles.

## 9. Grouping and aggregation

### 9.1 Sales by region

In [ ]:
sales_by_region = (
    df.groupby("region", dropna=False)
      .agg(
          total_sales=("sales", "sum"),
          average_sales=("sales", "mean"),
          order_count=("order_id", "count")
      )
      .sort_values("total_sales", ascending=False)
)

sales_by_region

In [ ]:
sales_by_region["total_sales"].plot(
    kind="bar",
    title="Total sales by region",
    xlabel="Region",
    ylabel="Total sales"
)
plt.xticks(rotation=0)
plt.show()

### 9.2 Sales by category and channel

In [ ]:
category_channel_summary = (
    df.groupby(["category", "channel"])
      .agg(
          total_sales=("sales", "sum"),
          average_discount=("discount", "mean"),
          orders=("order_id", "count")
      )
      .round(2)
)

category_channel_summary

### 9.3 Pivot table

In [ ]:
sales_pivot = pd.pivot_table(
    df,
    values="sales",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0
).round(2)

sales_pivot

Pivot tables are useful when an analyst needs a matrix-style summary across two categorical dimensions.

## 10. Analyze relationships between numerical variables

### 10.1 Scatter plot: unit price and sales

In [ ]:
df.plot(
    kind="scatter",
    x="unit_price",
    y="sales",
    title="Unit price versus order sales"
)
plt.show()

A positive relationship is expected because order sales are partly calculated from unit price. Quantity and discount also influence the final value.

### 10.2 Correlation matrix

In [ ]:
numeric_columns = [
    "unit_price",
    "quantity",
    "discount",
    "customer_rating",
    "delivery_days",
    "sales"
]

correlation_matrix = df[numeric_columns].corr()
correlation_matrix.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation_matrix, aspect="auto")

ax.set_xticks(range(len(numeric_columns)))
ax.set_xticklabels(numeric_columns, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_columns)))
ax.set_yticklabels(numeric_columns)

for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        ax.text(j, i, f"{correlation_matrix.iloc[i, j]:.2f}", ha="center", va="center")

ax.set_title("Correlation matrix")
fig.colorbar(image)
plt.tight_layout()
plt.show()

### Correlation interpretation

- values close to `+1` indicate a strong positive linear relationship;
- values close to `-1` indicate a strong negative linear relationship;
- values near `0` indicate little linear relationship.

Correlation does not establish causation. A high correlation may result from a direct relationship, a shared cause, a mathematical dependency, or chance.

## 11. Compare categorical and numerical variables

### 11.1 Sales distribution by category

In [ ]:
categories_order = ["Electronics", "Office", "Home", "Accessories"]
grouped_sales = [
    df.loc[df["category"] == category, "sales"].dropna()
    for category in categories_order
]

plt.boxplot(grouped_sales, labels=categories_order)
plt.title("Sales distribution by category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=15)
plt.show()

The box plot helps compare medians, variability, and extreme values across product categories.

### 11.2 Average rating by return status

In [ ]:
rating_by_return = (
    df.groupby("returned")["customer_rating"]
      .agg(["mean", "median", "count"])
      .round(2)
)

rating_by_return

This comparison may suggest an association between customer experience and returns, but further analysis would be needed before drawing a causal conclusion.

## 12. Compare two categorical variables

In [ ]:
return_by_channel = pd.crosstab(
    df["channel"],
    df["returned"],
    normalize="index"
).mul(100).round(2)

return_by_channel

A normalized crosstab reports the return-rate distribution within each sales channel.

## 13. Detect potential outliers with the IQR rule

In [ ]:
def iqr_outlier_report(series: pd.Series) -> dict:
    clean_series = series.dropna()
    q1 = clean_series.quantile(0.25)
    q3 = clean_series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (clean_series < lower) | (clean_series > upper)

    return {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum())
    }

iqr_report = pd.DataFrame({
    column: iqr_outlier_report(df[column])
    for column in ["unit_price", "quantity", "discount", "delivery_days", "sales"]
}).T

iqr_report.round(2)

In [ ]:
sales_q1 = df["sales"].quantile(0.25)
sales_q3 = df["sales"].quantile(0.75)
sales_iqr = sales_q3 - sales_q1
sales_upper = sales_q3 + 1.5 * sales_iqr

sales_outliers = df[df["sales"] > sales_upper].sort_values("sales", ascending=False)
sales_outliers.head(10)

Potential outliers should be investigated using domain knowledge. Removing them without justification may eliminate important business cases.

## 14. Examine missing-value patterns

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_by_column[missing_by_column > 0].plot(
    kind="bar",
    title="Missing values by column",
    xlabel="Column",
    ylabel="Missing count"
)
plt.xticks(rotation=30)
plt.show()

This chart shows which variables require attention. It does not explain why values are missing.

## 15. Create a compact data-quality report

In [ ]:
quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
    "constant_column": df.nunique(dropna=False).eq(1),
    "high_cardinality": df.nunique(dropna=True).gt(len(df) * 0.5)
})

quality_report

In [ ]:
dataset_quality_summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "columns_with_missing_values": int((df.isna().sum() > 0).sum()),
    "total_missing_cells": int(df.isna().sum().sum()),
    "constant_columns": quality_report.index[quality_report["constant_column"]].tolist(),
    "high_cardinality_columns": quality_report.index[quality_report["high_cardinality"]].tolist()
}

dataset_quality_summary

The `order_id` column is expected to have high cardinality because it is an identifier. High cardinality is therefore not automatically a problem.

## 16. Produce evidence-based findings

In [ ]:
top_region = sales_by_region["total_sales"].idxmax()
top_region_sales = sales_by_region.loc[top_region, "total_sales"]

top_category = df.groupby("category")["sales"].sum().idxmax()
top_category_sales = df.groupby("category")["sales"].sum().max()

return_rate = (df["returned"].eq("Yes").mean() * 100)

print(f"Highest-sales region: {top_region} ({top_region_sales:,.2f})")
print(f"Highest-sales category: {top_category} ({top_category_sales:,.2f})")
print(f"Overall return rate: {return_rate:.2f}%")
print(f"Median order sales: {df['sales'].median():,.2f}")

### Example interpretation

A valid EDA conclusion should connect a statement to measurable evidence.

Examples:

- The region with the highest total sales can be identified from the grouped summary.
- The difference between mean and median sales indicates right-skewness.
- Electronics has higher sales variability because unit prices are larger.
- Return rates vary across sales channels, but this pattern requires formal testing before a general conclusion is made.
- Missing values are limited, but they should still be treated before modeling.

Avoid conclusions that are not supported by the dataset.

## 17. Common EDA mistakes

1. Removing outliers automatically.
2. Treating correlation as causation.
3. Ignoring missing-value patterns.
4. Reporting charts without interpretation.
5. Using only averages and ignoring distribution shape.
6. Failing to distinguish identifiers from predictive features.
7. Performing analysis before checking duplicates and data types.
8. Generating many plots without linking them to analytical questions.

## 18. Guided exercises

1. Identify the three strongest absolute correlations excluding self-correlation.
2. Compare average sales between returned and non-returned orders.
3. Calculate the return rate for each region.
4. Create a monthly sales summary using `order_date`.
5. Compare median delivery time across sales channels.
6. Detect sales outliers using a z-score approach and compare the result with the IQR method.
7. Remove the duplicated row and recompute total sales.
8. Write three evidence-based findings and one analytical limitation.
9. Create a new feature called `discount_amount`.
10. Build a pivot table of average customer rating by region and channel.

## 19. Optional challenge: monthly trend analysis

In [ ]:
monthly_sales = (
    df.assign(month=df["order_date"].dt.to_period("M").astype(str))
      .groupby("month")["sales"]
      .sum()
)

monthly_sales.plot(
    kind="line",
    marker="o",
    title="Monthly sales trend",
    xlabel="Month",
    ylabel="Total sales"
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 20. Key conclusions

- EDA is a structured investigation of data quality, distributions, relationships, and unusual observations.
- Descriptive statistics and visualizations should be interpreted together.
- Grouping and pivot tables reveal differences across categories.
- Correlation measures linear association but does not establish causality.
- Outliers require investigation rather than automatic removal.
- Data-quality reporting should precede modeling.
- Strong EDA communicates findings, evidence, uncertainty, and limitations.